In [ ]:
# !pip install ninja ipykernel ipywidgets --break-system-packages

In [ ]:
# !pip install --upgrade transformers --break-system-packages

In [ ]:
# !pip install git+https://github.com/intel/auto-round.git --break-system-packages

In [5]:
import os
import torch
from auto_round import AutoRound
from huggingface_hub import HfApi, create_repo, notebook_login, get_token
from transformers import AutoModelForImageTextToText, AutoProcessor

In [6]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [7]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch Version: 2.12.0+cu130
CUDA Available: True
CUDA Version: 13.0
GPU Name: NVIDIA H100 PCIe
VRAM: 79.2 GB


In [8]:
notebook_login()

In [9]:
MODEL_ID = "meta-models/Muse-Glimmer-30B"
HF_USER = "Vishva007"
OUTPUT_BASE_DIR = "./AutoRound"
LOCAL_PATH = "./local_model"

In [ ]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

In [10]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, 
    dtype=torch.bfloat16, 
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/1436 [00:00<?, ?it/s]

In [11]:
model

MuseGlimmerForConditionalGeneration(
  (model): MuseGlimmerModel(
    (vision_tower): MuseGlimmerVisionModel(
      (patch_embedder): MuseGlimmerVisionPatchEmbedder(
        (patch_embedding): Linear(in_features=1176, out_features=1536, bias=False)
        (position_embedding_table): Embedding(1024, 1536)
      )
      (rotary_emb): MuseGlimmerVisionRotaryEmbedding()
      (ln_pre): LayerNorm((1536,), eps=1e-05, elementwise_affine=True, bias=True)
      (layers): ModuleList(
        (0-49): 50 x MuseGlimmerVisionEncoderLayer(
          (norm1): LayerNorm((1536,), eps=1e-05, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((1536,), eps=1e-05, elementwise_affine=True, bias=True)
          (attn): MuseGlimmerVisionAttention(
            (proj): Linear(in_features=1536, out_features=1536, bias=True)
            (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
            (k_proj): Linear(in_features=1536, out_features=1536, bias=True)
            (v_proj): L

In [13]:
TUNING_CONFIG = {
    "group_size": 32,
    "sym": True,
    "iters": 1000,  # High accuracy (Production grade)
    "nsamples": 512,  # More calibration data
    "batch_size": 2,  # Faster on 48GB VRAM
    "seqlen": 2048,
    "low_gpu_mem_usage": False,  # Keep on GPU for speed
    "enable_torch_compile": True,  # JIT acceleration
    "quant_nontext_module": False,  # Keep Vision Tower in FP16 (Crucial for VLM accuracy)
}

In [14]:
def push_to_hub(local_dir, repo_name, token):
    """Creates repo and uploads folder to Hugging Face."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")

    try:
        api = HfApi()
        create_repo(
            full_repo_id, repo_type="model", exist_ok=True, private=False, token=token
        )

        api.upload_folder(
            folder_path=local_dir, repo_id=full_repo_id, repo_type="model", token=token
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:  # noqa: BLE001
        print(f"[Hub] ❌ Error uploading: {e}")

In [15]:
ar = AutoRound(
    model=model,
    tokenizer=tokenizer,
    processor=processor,
    scheme="W2A16",
    **TUNING_CONFIG,
)

2026-08-11 15:46:05 WARNING autoround.py L551: Passing 'group_size' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-08-11 15:46:05 WARNING autoround.py L551: Passing 'sym' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-08-11 15:46:05 WARNING autoround.py L551: Passing 'iters' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.


In [16]:
# SINGLE CALL to save all 3 formats to the same output directory
# The files will exist side-by-side or merged in this folder.
ar.quantize_and_save(
    OUTPUT_BASE_DIR, format="auto_round,auto_gptq", inplace=True
)

2026-08-11 15:46:08 INFO config.py L112: set the lr to 2.0/iters for better accuracy
2026-08-11 15:46:08 WARNING logging.py L340: some layers are skipped quantization (shape not divisible by 32): model.vision_tower.patch_embedder.patch_embedding
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
2026-08-11 15:46:10 INFO orchestrator.py L570: start to cache block inputs
2026-08-11 15:46:10 INFO mllm.py L86: Using MLLM template: muse_glimmer
2026-08-11 15:46:10 INFO calib_dataset.py L1107: Preprocessing calibration dataset in a subprocess to avoid memory leaks...
2026-08-11 15:47:50 INFO device.py L1448: 'peak_ram': 72.16GB, 'peak_vram': 55.83GB
2026-08-11 15:47:50 INFO orchestrator.py L602: caching done
Quantizing model.language_model.layers.0:   0%|          | 0/52 [00:04<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:882: UserWarning: cuDNN Attention defaults to a non-deterministic algorith

Writing model shards:   0%|          | 0/4 [00:00<?, ?it/s]

packing: 100%|██████████| 417/417 [01:37<00:00,  4.28it/s]


Writing model shards:   0%|          | 0/4 [00:00<?, ?it/s]

2026-08-11 16:45:11 INFO device.py L1448: 'peak_ram': 72.51GB, 'peak_vram': 55.83GB


(MuseGlimmerForConditionalGeneration(
   (model): MuseGlimmerModel(
     (vision_tower): MuseGlimmerVisionModel(
       (patch_embedder): MuseGlimmerVisionPatchEmbedder(
         (patch_embedding): Linear(in_features=1176, out_features=1536, bias=False)
         (position_embedding_table): Embedding(1024, 1536)
       )
       (rotary_emb): MuseGlimmerVisionRotaryEmbedding()
       (ln_pre): LayerNorm((1536,), eps=1e-05, elementwise_affine=True, bias=True)
       (layers): ModuleList(
         (0-49): 50 x MuseGlimmerVisionEncoderLayer(
           (norm1): LayerNorm((1536,), eps=1e-05, elementwise_affine=True, bias=True)
           (norm2): LayerNorm((1536,), eps=1e-05, elementwise_affine=True, bias=True)
           (attn): MuseGlimmerVisionAttention(
             (proj): Linear(in_features=1536, out_features=1536, bias=True)
             (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
             (k_proj): Linear(in_features=1536, out_features=1536, bias=True)
      

In [17]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [19]:
if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w2g32/auto-round-auto-gptq"), 
        f"{base_name}-W2A16-AutoRound", 
        hf_token)
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w2g32/auto-gptq"), 
        f"{base_name}-W2A16-AutoRound-GPTQ",
        hf_token
    )
else:
    print("No Hugging Face token found. Skipping upload to hub.")


[Hub] Pushing ./AutoRound/local_model-w2g32/auto-round-auto-gptq to Vishva007/Muse-Glimmer-30B-W2A16-AutoRound...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/Muse-Glimmer-30B-W2A16-AutoRound

[Hub] Pushing ./AutoRound/local_model-w2g32/auto-gptq to Vishva007/Muse-Glimmer-30B-W2A16-AutoRound-GPTQ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/Muse-Glimmer-30B-W2A16-AutoRound-GPTQ
